# RAGBench (finqa) — Final Pipeline
**Project:** Evaluating the Reliability of Selected RAG Evaluation Metrics for Finance-Related QA

## 1. Install a compatible, pinned set of libraries

In [ ]:
!pip install -q \
  "ragas==0.2.15" \
  "langchain==0.3.27" "langchain-core==0.3.76" "langchain-community==0.3.30" \
  "langchain-openai==0.2.14" "langchain-groq==0.2.4" "langchain-huggingface==0.1.2" \
  datasets sentence-transformers

print("Libraries installed.")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 190.9/190.9 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 42.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 447.5/447.5 kB 42.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 948.6/948.6 kB 62.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 5.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the sou

## 2. Add your API keys

In [ ]:
import os
from getpass import getpass

os.environ["GROQ_API_KEY"] = getpass("Paste Groq API key here: ")
print("Groq key stored for this session.")


Paste Groq API key here: ··········
Groq key stored for this session.


In [ ]:
from huggingface_hub import login

login(getpass("Paste Hugging Face token here: "))


Paste Hugging Face token here: ··········


## 3. Load finqa and take the sample

`random_state=42` fixes the sample so it is reproducible — the same 15 rows every time
this is run.


In [ ]:
from datasets import load_dataset
import pandas as pd

ragbench_finqa = load_dataset("rungalileo/ragbench", "finqa")
df_finqa = ragbench_finqa["train"].to_pandas()

SAMPLE_SIZE = 15
df_sample = df_finqa.sample(n=SAMPLE_SIZE, random_state=42).reset_index(drop=True)

print("Sample shape:", df_sample.shape)
df_sample[["id", "question", "response"]].head()


README.md:   0%|          | 0.00/24.7k [00:00<?, ?B/s]

finqa/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 61.1MB            

finqa/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

finqa/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 5.94MB            

finqa/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

Generating train split:   0%|          | 0/12502 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1766 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2294 [00:00<?, ? examples/s]

Sample shape: (15, 26)


,id,question,response
0,finqa_4895,what percent of the total future obligations i...,To find the percent of the total future obliga...
1,finqa_1253,what is the range between the shortest and lon...,Based on the information provided in the conte...
2,finqa_3181,in 2007 what was the percentage change in the ...,According to the reconciliation of the unrecog...
3,finqa_2743,"with 2014 closing stock price , what is the to...",Based on the information provided in the conte...
4,finqa_490,what was the ratio of the outstanding surety b...,The outstanding surety bonds for the investee ...


## 4. Tag each question (four-category scheme)

Expanded from the original two categories (calculation/lookup) to four, so the same
framework can be applied consistently across all three datasets:

- **Lookup** — single factual retrieval, no computation required
- **Calculation** — requires arithmetic (percentage, ratio, difference, growth, etc.)
- **Comparison** — explicitly evaluates or contrasts two or more entities/values
- **Multi-hop** — requires combining evidence from multiple distinct sources/passages

Priority order when a question could match more than one (e.g. a comparison that also
involves a calculation): **multi-hop > comparison > calculation > lookup**. This is a
judgment call worth stating explicitly in the methodology, not a hidden detail.

In [ ]:
MULTIHOP_KEYWORDS = [
    "based on both", "combining", "across all", "in both", "considering all"
]

COMPARISON_KEYWORDS = [
    "compare", "compared", "comparison", "versus", " vs ",
    "higher than", "lower than", "greater than", "which is more", "which is higher"
]

CALCULATION_KEYWORDS = [
    "percent", "percentage", "change", "ratio", "difference",
    "growth", "increase", "decrease", "average", "rate of"
]

def tag_question_type(question):
    q = question.lower()
    if any(keyword in q for keyword in MULTIHOP_KEYWORDS):
        return "multi-hop"
    if any(keyword in q for keyword in COMPARISON_KEYWORDS):
        return "comparison"
    if any(keyword in q for keyword in CALCULATION_KEYWORDS):
        return "calculation"
    return "lookup"

def total_context_length(documents):
    return sum(len(doc) for doc in documents)

df_sample["question_type"] = df_sample["question"].apply(tag_question_type)
df_sample["context_length"] = df_sample["documents"].apply(total_context_length)

print(df_sample["question_type"].value_counts())

question_type
calculation    9
lookup         6
Name: count, dtype: int64


## 5. Set up the LLM judge and embedding model

- LLM judge: Groq running Llama 3.1 (free, fast, hosted — no GPU needed locally)
- Embeddings: a small sentence-transformers model, runs locally on CPU


In [ ]:
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

judge_llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0)
ragas_llm = LangchainLLMWrapper(judge_llm)

embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
ragas_embeddings = LangchainEmbeddingsWrapper(embedding_model)

print("LLM judge and embeddings ready.")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

LLM judge and embeddings ready.


## 6. Run the three RAGAS metrics

- **Faithfulness** — is the response grounded in the retrieved context?
- **Answer Relevancy** — does the response actually address the question?
- **Context Precision** (reference-free variant — finqa has no separate ground-truth
  answer, only the response being evaluated, so this variant judges precision using
  the response itself)

Run slowly (2 requests at a time) to stay within Groq's free-tier rate limits.
Expect roughly 15-20 minutes for 15 rows. Occasional individual timeouts are normal
for a free-tier API — Cell 7 checks for and handles these.


In [ ]:
from ragas import evaluate, EvaluationDataset
from ragas.metrics import faithfulness, answer_relevancy, LLMContextPrecisionWithoutReference
from ragas.run_config import RunConfig

context_precision_metric = LLMContextPrecisionWithoutReference(llm=ragas_llm)

slow_and_steady = RunConfig(
    timeout=300,
    max_workers=2,
    max_retries=15,
    max_wait=90,
)

ragas_rows = []
for _, row in df_sample.iterrows():
    ragas_rows.append({
        "user_input": row["question"],
        "response": row["response"],
        "retrieved_contexts": list(row["documents"]),
    })

eval_dataset = EvaluationDataset.from_list(ragas_rows)

results = evaluate(
    dataset=eval_dataset,
    metrics=[faithfulness, answer_relevancy, context_precision_metric],
    llm=ragas_llm,
    embeddings=ragas_embeddings,
    run_config=slow_and_steady,
)

results_df = results.to_pandas()
results_df


Evaluating:   0%|          | 0/45 [00:00<?, ?it/s]

ERROR:ragas.executor:Exception raised in Job[9]: TimeoutError()
ERROR:ragas.executor:Exception raised in Job[36]: TimeoutError()


,user_input,retrieved_contexts,response,faithfulness,answer_relevancy,llm_context_precision_without_reference
0,what percent of the total future obligations i...,[23t . rowe price group | annual report 2013 c...,To find the percent of the total future obliga...,0.923077,1.000000,1.000000
1,what is the range between the shortest and lon...,[notes to consolidated financial statements ( ...,Based on the information provided in the conte...,0.714286,0.980723,1.000000
2,in 2007 what was the percentage change in the ...,[determined that it will primarily be subject ...,According to the reconciliation of the unrecog...,0.400000,0.898600,1.000000
3,"with 2014 closing stock price , what is the to...","[part iii item 10 . directors , executive offi...",Based on the information provided in the conte...,NaN,0.764775,1.000000
4,what was the ratio of the outstanding surety b...,[at december 31 . the following table summariz...,The outstanding surety bonds for the investee ...,0.500000,0.932229,0.333333
5,what is the change in value of fixed maturitie...,"[the hartford financial services group , inc ....",The change in value of fixed maturities AFS fr...,0.800000,0.925963,0.000000
6,what was the percentage change of the net reve...,[entergy corporation and subsidiaries manageme...,"The net revenue in 2010 was $5051 million, and...",0.312500,0.931566,1.000000
7,what portion of the total accumulated other co...,[note 17 . accumulated other comprehensive los...,"According to the provided context, the currenc...",0.750000,0.854197,1.000000
8,what percent did indemnified securities financ...,[state street bank issuances : state street ba...,Based on the information provided in the conte...,0.500000,0.986001,0.583333
9,what is the total unfunded commitments at dece...,[whether or not any claims asserted against us...,"According to the context provided, the total u...",0.625000,0.838909,1.000000


## 7. Check for and retry any failed rows

Free-tier APIs occasionally time out on individual requests — this is normal.
Failed scores show up as missing (NaN). We check how many, and retry just those.


In [ ]:
metric_cols = ["faithfulness", "answer_relevancy", "llm_context_precision_without_reference"]

missing_counts = results_df[metric_cols].isna().sum()
print("Missing values per metric:")
print(missing_counts)


Missing values per metric:
faithfulness                               2
answer_relevancy                           0
llm_context_precision_without_reference    0
dtype: int64


In [ ]:
failed_mask = results_df[metric_cols].isna().any(axis=1)
failed_indices = results_df[failed_mask].index.tolist()

print(f"Retrying {len(failed_indices)} row(s): {failed_indices}")

if len(failed_indices) > 0:
    retry_rows = [ragas_rows[i] for i in failed_indices]
    retry_dataset = EvaluationDataset.from_list(retry_rows)

    retry_results = evaluate(
        dataset=retry_dataset,
        metrics=[faithfulness, answer_relevancy, context_precision_metric],
        llm=ragas_llm,
        embeddings=ragas_embeddings,
        run_config=slow_and_steady,
    )
    retry_df = retry_results.to_pandas()

    for pos, orig_idx in enumerate(failed_indices):
        for col in metric_cols:
            if pd.isna(results_df.loc[orig_idx, col]):
                results_df.loc[orig_idx, col] = retry_df.loc[pos, col]

    print("Retry complete. Remaining missing values:")
    print(results_df[metric_cols].isna().sum())
else:
    print("No missing values — nothing to retry.")


Retrying 2 row(s): [3, 12]


Evaluating:   0%|          | 0/6 [00:00<?, ?it/s]

Retry complete. Remaining missing values:
faithfulness                               0
answer_relevancy                           0
llm_context_precision_without_reference    0
dtype: int64


## 8. Build the final results table and save it

In [ ]:
comparison_df = pd.concat([
    df_sample[["id", "question", "question_type", "context_length",
               "adherence_score", "relevance_score"]].reset_index(drop=True),
    results_df[metric_cols].reset_index(drop=True)
], axis=1)

comparison_df.to_csv("ragbench_finqa_final_results.csv", index=False)
print("Saved: ragbench_finqa_final_results.csv")
comparison_df


Saved: ragbench_finqa_final_results.csv


,id,question,question_type,context_length,adherence_score,relevance_score,faithfulness,answer_relevancy,llm_context_precision_without_reference
0,finqa_4895,what percent of the total future obligations i...,calculation,4822,True,0.041667,0.923077,1.000000,1.000000
1,finqa_1253,what is the range between the shortest and lon...,lookup,2814,True,0.058824,0.714286,0.980723,1.000000
2,finqa_3181,in 2007 what was the percentage change in the ...,calculation,4284,True,0.035714,0.400000,0.898600,1.000000
3,finqa_2743,"with 2014 closing stock price , what is the to...",lookup,9124,True,0.108108,1.000000,0.764775,1.000000
4,finqa_490,what was the ratio of the outstanding surety b...,calculation,3600,True,0.040000,0.500000,0.932229,0.333333
5,finqa_3697,what is the change in value of fixed maturitie...,calculation,2083,True,0.076923,0.800000,0.925963,0.000000
6,finqa_4784,what was the percentage change of the net reve...,calculation,2976,True,0.071429,0.312500,0.931566,1.000000
7,finqa_3189,what portion of the total accumulated other co...,lookup,3608,True,0.045455,0.750000,0.854197,1.000000
8,finqa_1993,what percent did indemnified securities financ...,calculation,4254,True,0.037037,0.500000,0.986001,0.583333
9,finqa_4566,what is the total unfunded commitments at dece...,lookup,5759,True,0.033333,0.625000,0.838909,1.000000


## 9. Compare average scores by question type

The main finding: does Faithfulness / Context Precision differ between calculation-based
and direct-lookup finance questions?


In [ ]:
print("Average scores by question type:")
print(comparison_df.groupby("question_type")[metric_cols].mean().round(3))
print()
print("Row counts per group:")
print(comparison_df["question_type"].value_counts())


Average scores by question type:
               faithfulness  answer_relevancy  \
question_type                                   
calculation           0.590             0.941   
lookup                0.732             0.797   

               llm_context_precision_without_reference  
question_type                                           
calculation                                      0.722  
lookup                                           1.000  

Row counts per group:
question_type
calculation    9
lookup         6
Name: count, dtype: int64


## 10. Check correlation with context length

In [ ]:
print("Correlation between context_length and each metric:")
print(comparison_df[["context_length"] + metric_cols].corr()["context_length"].round(3))


Correlation between context_length and each metric:
context_length                             1.000
faithfulness                               0.434
answer_relevancy                          -0.210
llm_context_precision_without_reference    0.301
Name: context_length, dtype: float64


## 11. Qualitative review of the three flagged rows

Reading the full question, context, response, and RAGBench's own annotator
explanations for the most interesting disagreements found in this sample.


In [ ]:
import textwrap

rows_to_inspect = ["finqa_4784", "finqa_3515", "finqa_3697"]

WRAP_WIDTH = 70

for row_id in rows_to_inspect:
    if row_id not in df_sample["id"].values:
        continue
    row = df_finqa[df_finqa["id"] == row_id].iloc[0]
    scored = comparison_df[comparison_df["id"] == row_id].iloc[0]

    print("=" * WRAP_WIDTH)
    print(f"ID: {row_id}")
    print("=" * WRAP_WIDTH)
    print("QUESTION:")
    print(textwrap.fill(row["question"], WRAP_WIDTH))
    print()
    print("DOCUMENTS (retrieved context):")
    for i, doc in enumerate(row["documents"]):
        snippet = doc[:300] + "..." if len(doc) > 300 else doc
        print(f"[{i}]")
        print(textwrap.fill(snippet, WRAP_WIDTH))
        print()
    print("RESPONSE (being evaluated):")
    print(textwrap.fill(row["response"], WRAP_WIDTH))
    print()
    print(f"Faithfulness: {scored['faithfulness']:.3f}   Context Precision: {scored['llm_context_precision_without_reference']:.3f}")
    print(f"Human adherence_score: {row['adherence_score']}")


ID: finqa_4784
QUESTION:
what was the percentage change of the net revenue in 2010

DOCUMENTS (retrieved context):
[0]
entergy corporation and subsidiaries management's financial discussion
and analysis refer to 201cselected financial data - five-year
comparison of entergy corporation and subsidiaries 201d which
accompanies entergy corporation 2019s financial statements in this
report for further information with re...

[1]
[["", "amount ( in millions )"], ["2009 net revenue", "$ 4694"],
["volume/weather", "231"], ["retail electric price", "137"],
["provision for regulatory proceedings", "26"], ["rough production
cost equalization", "19"], ["ano decommissioning trust", "-24 ( 24
)"], ["fuel recovery", "-44 ( 44 )"], ["...

[2]
the volume/weather variance is primarily due to an increase of 8362
gwh , or 8% ( 8 % ) , in billed electricity usage in all retail
sectors , including the effect on the residential sector of colder
weather in the first quarter 2010 compared to 2009 and warmer we

## Summary of findings (finqa, n=15)

- Calculation-based questions scored lower on average than lookup questions for both
  Faithfulness and Context Precision, despite most responses being rated "adherent" by
  human/GPT-4o annotators.
- Close reading suggests Faithfulness may penalise correctly derived numerical answers
  that are not stated verbatim in the source text, and Context Precision may fail to
  recognise tabular data as relevant context.
- Sample size (n=15) is small, constrained by free-tier API rate limits — stated here as
  an explicit limitation. This is an indicative pattern, not a statistically confirmed one.

**Next dataset: MultiHop-RAG, using the same pipeline structure.**
